In [ ]:
"""
Uso:
    python abp_6.py

Salidas:
    - graficos/  → PNG por lección
    - informe_final.pdf
    - diccionario_variables.txt
"""

import os
import sys
import warnings
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import norm, binom, poisson

warnings.filterwarnings('ignore')

try:
    sys.stdout.reconfigure(encoding='utf-8')
except Exception:
    pass

BASE_DIR = Path(__file__).resolve().parent
DEFAULT_DATASET = BASE_DIR / "data" / "dataset_simulado.csv"

# ── Paleta de colores ────────────────────────────────────────────────────────
NAVY   = "#1F4E79"
BLUE   = "#2E86C1"
TEAL   = "#17A589"
RED    = "#C0392B"
AMBER  = "#D4AC0D"
GRAY   = "#566573"
LGRAY  = "#F2F3F4"
WHITE  = "#FFFFFF"

os.makedirs("graficos", exist_ok=True)


def resolve_dataset_path() -> Path:
    """Devuelve la ruta fija del dataset simulado si existe."""
    if DEFAULT_DATASET.exists():
        return DEFAULT_DATASET

    raise FileNotFoundError(
        "No se encontró el dataset requerido.\n"
        f"Ruta esperada: {DEFAULT_DATASET}\n"
        "Este script trabaja únicamente con dataset_simulado.csv en la carpeta data/."
    )


def require_columns(df: pd.DataFrame, columns: list[str]) -> None:
    """Frena la ejecución con un mensaje claro si faltan columnas."""
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise KeyError(
            "Faltan columnas obligatorias en el dataset: "
            + ", ".join(missing)
            + f". Columnas disponibles: {list(df.columns)}"
        )


def safe_std(data: np.ndarray | pd.Series, ddof: int = 1) -> float:
    """Calcula desviación estándar evitando NaN en muestras muy pequeñas."""
    data = pd.Series(data).dropna()
    if len(data) <= ddof:
        return 0.0
    return float(np.std(data, ddof=ddof))


def ic(data, conf):
    """Calcula IC para la media con t-Student."""
    data = pd.Series(data).dropna().astype(float)
    n_ = len(data)
    if n_ == 0:
        raise ValueError("No hay datos válidos para calcular el intervalo de confianza.")
    xbar = float(np.mean(data))
    s = safe_std(data, ddof=1)
    se = s / np.sqrt(n_) if n_ > 0 else 0.0
    if n_ < 2 or se == 0:
        return xbar, s, n_, se, np.nan, xbar, xbar
    t = stats.t.ppf((1 + conf) / 2, df=n_ - 1)
    return xbar, s, n_, se, t, xbar - t * se, xbar + t * se


# ==========================================================================
# CARGA DEL DATASET
# ==========================================================================
print("=" * 60)
print("CARGANDO DATASET...")
data_path = resolve_dataset_path()
print(f"  Archivo: {data_path}")
df = pd.read_csv(filepath_or_buffer=str(data_path))
n = len(df)
print(f"  Registros: {n} | Variables: {df.shape[1]}")
print()

require_columns(
    df,
    [
        "calidad_sueno",
        "nivel_af",
        "psqi_total",
        "horas_sueno",
        "latencia_min",
        "eficiencia_pct",
        "perturbaciones",
        "disfuncion_diurna",
        "mets_semana",
    ],
)

# Normalización ligera de texto para evitar errores por espacios accidentales
for col in ["calidad_sueno", "nivel_af"]:
    if df[col].dtype == object:
        df[col] = df[col].astype(str).str.strip()

# ==========================================================================
# DICCIONARIO DE VARIABLES
# ==========================================================================
diccionario = """
============================================================
DICCIONARIO DE VARIABLES — dataset_simulado.csv
============================================================
Variable             Tipo          Escala       Descripción
------------------------------------------------------------
id                   Cualitativa   Nominal      Identificador único del estudiante (EST001…EST100)
edad                 Cuantitativa  Discreta     Edad en años cumplidos (18–25)
sexo                 Cualitativa   Nominal      Femenino / Masculino
carrera              Cualitativa   Nominal      Carrera universitaria (8 categorías)
horas_sueno          Cuantitativa  Continua     Horas dormidas por noche (aprox.)
latencia_min         Cuantitativa  Continua     Minutos para conciliar el sueño
eficiencia_pct       Cuantitativa  Continua     % eficiencia del sueño (tiempo dormido/en cama)
calidad_subjetiva    Cuantitativa  Ordinal      Calidad subjetiva PSQI (0=muy buena … 3=muy mala)
perturbaciones       Cuantitativa  Ordinal      Perturbaciones del sueño PSQI (0–3)
uso_medicacion       Cuantitativa  Ordinal      Uso de medicación para dormir PSQI (0–3)
disfuncion_diurna    Cuantitativa  Ordinal      Disfunción diurna PSQI (0–3)
psqi_total           Cuantitativa  Discreta     Puntaje total PSQI (0–21); >5 = Mal dormidor
calidad_sueno        Cualitativa   Nominal      Buen dormidor (PSQI≤5) / Mal dormidor (PSQI>5)
af_vigorosa_dias     Cuantitativa  Discreta     Días/semana de AF vigorosa
af_moderada_dias     Cuantitativa  Discreta     Días/semana de AF moderada
caminata_dias        Cuantitativa  Discreta     Días/semana de caminata ≥10 min
mets_semana          Cuantitativa  Continua     METs·min·semana totales
nivel_af             Cualitativa   Ordinal      Bajo(≤600) / Moderado(601–2999) / Alto(≥3000 METs)
============================================================
"""
with open("diccionario_variables.txt", "w", encoding="utf-8") as f:
    f.write(diccionario)
print(diccionario)


# ==========================================================================
# LECCIÓN 1 — MÉTODO CIENTÍFICO: Descriptivos y Variables
# ==========================================================================
print("=" * 60)
print("L1 — MÉTODO CIENTÍFICO")
print("=" * 60)

print("\n► Distribución de Calidad de Sueño:")
cs_counts = df["calidad_sueno"].fillna("Sin dato").value_counts()
for k, v in cs_counts.items():
    print(f"   {k}: {v} ({v/n*100:.1f}%)")

print("\n► Distribución de Nivel AF:")
af_counts = df["nivel_af"].fillna("Sin dato").value_counts()
for k, v in af_counts.items():
    print(f"   {k}: {v} ({v/n*100:.1f}%)")

print("\n► Estadísticos descriptivos — Variables cuantitativas clave:")
cols_desc = ["psqi_total", "horas_sueno", "latencia_min", "eficiencia_pct", "mets_semana"]
desc = df[cols_desc].describe().round(2)
print(desc.to_string())

# Gráfico L1
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("L1 · Descripción General de la Muestra", fontsize=13, fontweight="bold", color=NAVY)

# CS
ax = axes[0]
labels = cs_counts.index.tolist()
sizes  = cs_counts.values
colors = [RED, TEAL]
wedges, texts, autotexts = ax.pie(sizes, labels=labels, colors=colors,
                                   autopct="%1.1f%%", startangle=90,
                                   textprops={"fontsize": 10})
for at in autotexts:
    at.set_color(WHITE); at.set_fontweight("bold")
ax.set_title("Calidad de Sueño", fontsize=11, color=NAVY, pad=10)

# AF
ax = axes[1]
order = ["Alto", "Moderado", "Bajo"]
vals  = [af_counts.get(o, 0) for o in order]
bars  = ax.barh(order, vals, color=[TEAL, BLUE, RED], edgecolor="white", height=0.6)
for bar, v in zip(bars, vals):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f"{v} ({v/n*100:.0f}%)", va="center", fontsize=10, color=GRAY)
ax.set_xlim(0, max(max(vals), 1) * 1.3)
ax.set_xlabel("Frecuencia", fontsize=9)
ax.set_title("Nivel de Actividad Física", fontsize=11, color=NAVY)
ax.spines[["top", "right"]].set_visible(False)

# PSQI histogram
ax = axes[2]
ax.hist(df["psqi_total"].dropna(), bins=12, color=BLUE, edgecolor=WHITE, alpha=0.85)
ax.axvline(5, color=RED, linestyle="--", linewidth=1.5, label="Corte = 5")
ax.set_xlabel("Puntaje PSQI Total", fontsize=10)
ax.set_ylabel("Frecuencia", fontsize=10)
ax.set_title("Distribución Puntaje PSQI", fontsize=11, color=NAVY)
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("graficos/L1_descripcion.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n  ✓ graficos/L1_descripcion.png guardado")


# ==========================================================================
# LECCIÓN 2 — PROBABILIDADES
# ==========================================================================
print("\n" + "=" * 60)
print("L2 — PROBABILIDAD Y MUESTREO")
print("=" * 60)

p_mal   = (df["calidad_sueno"] == "Mal dormidor").mean()
p_bajo  = (df["nivel_af"] == "Bajo").mean()
p_inter = ((df["calidad_sueno"] == "Mal dormidor") & (df["nivel_af"] == "Bajo")).mean()
p_union = p_mal + p_bajo - p_inter
p_cond  = p_inter / p_mal if p_mal > 0 else np.nan
p_disf  = (df["disfuncion_diurna"] == 3).mean()

print(f"\n  P(Mal dormidor)              = {p_mal:.4f}  ({p_mal*100:.1f}%)")
print(f"  P(AF Bajo)                   = {p_bajo:.4f}  ({p_bajo*100:.1f}%)")
print(f"  P(Mal dorm ∩ AF Bajo)        = {p_inter:.4f}  ({p_inter*100:.1f}%)")
print(f"  P(Mal dorm ∪ AF Bajo)        = {p_union:.4f}  ({p_union*100:.1f}%)")
print(f"  P(AF Bajo | Mal dormidor)    = {p_cond:.4f}  ({p_cond*100:.1f}%)")
print(f"  P(Disfunción diurna severa)  = {p_disf:.4f}  ({p_disf*100:.1f}%)")

# Gráfico L2 — Árbol de probabilidad simplificado
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("L2 · Probabilidades Básicas", fontsize=13, fontweight="bold", color=NAVY)

# Barplot probabilidades
ax = axes[0]
eventos = ["P(Mal\ndormidor)", "P(AF Bajo)", "P(A∩B)", "P(A∪B)", "P(AF Bajo\n|Mal dorm.)"]
probs   = [p_mal, p_bajo, p_inter, p_union, p_cond]
colors_p = [RED, AMBER, NAVY, TEAL, BLUE]
bars = ax.bar(eventos, probs, color=colors_p, edgecolor=WHITE, width=0.6)
for bar, v in zip(bars, probs):
    if pd.notna(v):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{v:.3f}", ha="center", fontsize=9, fontweight="bold", color=GRAY)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Probabilidad", fontsize=10)
ax.set_title("Probabilidades Calculadas (n=" + str(n) + ")", fontsize=11, color=NAVY)
ax.spines[["top", "right"]].set_visible(False)
ax.axhline(0.5, color=GRAY, linestyle=":", linewidth=0.8, alpha=0.5)

# Árbol visual
ax = axes[1]
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis("off")
ax.set_title("Árbol de Probabilidad — CS × Nivel AF", fontsize=11, color=NAVY)

# Nodo raíz
ax.annotate("", xy=(3, 7), xytext=(1, 5),
            arrowprops=dict(arrowstyle="->", color=RED, lw=1.5))
ax.annotate("", xy=(3, 3), xytext=(1, 5),
            arrowprops=dict(arrowstyle="->", color=TEAL, lw=1.5))

ax.text(1, 5, f"n={n}", ha="center", va="center",
        fontsize=10, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.3", fc=LGRAY, ec=NAVY, lw=1.5))

ax.text(1.8, 6.3, f"P={p_mal:.2f}", fontsize=8, color=RED)
ax.text(1.8, 3.7, f"P={1-p_mal:.2f}", fontsize=8, color=TEAL)

ax.text(3.2, 7, f"Mal dormidor\n({int(p_mal*100)}%)", ha="left", va="center",
        fontsize=9, bbox=dict(boxstyle="round,pad=0.3", fc="#FADBD8", ec=RED, lw=1))
ax.text(3.2, 3, f"Buen dormidor\n({int((1-p_mal)*100)}%)", ha="left", va="center",
        fontsize=9, bbox=dict(boxstyle="round,pad=0.3", fc="#D1F2EB", ec=TEAL, lw=1))

# Ramas de Mal dormidor
for i, (label, p_val, col) in enumerate(zip(
    ["AF Alto", "AF Moderado", "AF Bajo"],
    [0.185, 0.370, 0.444], [TEAL, BLUE, RED])):
    y_dest = 8.5 - i * 1.5
    ax.annotate("", xy=(7, y_dest), xytext=(5.5, 7),
                arrowprops=dict(arrowstyle="->", color=col, lw=1.2))
    ax.text(7.1, y_dest, f"{label}\n{p_val:.3f}", ha="left", va="center",
            fontsize=8, color=col, fontweight="bold")

ax.text(6.3, 7.2, "dado Mal\ndormidor", fontsize=7, color=GRAY, ha="center")

plt.tight_layout()
plt.savefig("graficos/L2_probabilidades.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ graficos/L2_probabilidades.png guardado")


# ==========================================================================
# LECCIÓN 3 — DISTRIBUCIONES DE PROBABILIDAD
# ==========================================================================
print("\n" + "=" * 60)
print("L3 — DISTRIBUCIONES DE PROBABILIDAD")
print("=" * 60)

# Parámetros estimados del dataset
mu_psqi  = df["psqi_total"].mean()
sd_psqi  = safe_std(df["psqi_total"], ddof=1)
p_bin    = p_mal          # prob mal dormidor
lam_poi  = df["perturbaciones"].mean()

print(f"\n  Binomial: n={n}, p={p_bin:.4f}")
print(f"    E(X) = {n*p_bin:.2f}  |  SD = {np.sqrt(n*p_bin*(1-p_bin)):.2f}")
print(f"\n  Poisson: lambda = {lam_poi:.4f}")
print(f"    P(X=0) = {poisson.pmf(0,lam_poi):.4f}")
print(f"    P(X=1) = {poisson.pmf(1,lam_poi):.4f}")
print(f"    P(X>=2) = {1-poisson.cdf(1,lam_poi):.4f}")
print(f"\n  Normal PSQI: mu={mu_psqi:.2f}, sigma={sd_psqi:.2f}")
if sd_psqi > 0:
    print(f"    P(PSQI>5)  = {1-norm.cdf(5, mu_psqi, sd_psqi):.4f}")
    print(f"    P(PSQI>10) = {1-norm.cdf(10,mu_psqi, sd_psqi):.4f}")
else:
    print("    La desviación estándar es 0; no se puede ajustar una normal útil.")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle("L3 · Distribuciones de Probabilidad", fontsize=13, fontweight="bold", color=NAVY)

# Binomial
ax = axes[0]
x_b = np.arange(max(0, int(n*p_bin) - 15), min(n, int(n*p_bin) + 15)) if n > 0 else np.array([0])
y_b = binom.pmf(x_b, n, p_bin) if n > 0 else np.array([1.0])
ax.bar(x_b, y_b, color=NAVY, alpha=0.8, edgecolor=WHITE, width=0.7)
ax.axvline(n*p_bin, color=RED, linestyle="--", linewidth=1.5, label=f"E(X)={n*p_bin:.1f}")
ax.set_xlabel("Número de Mal Dormidores", fontsize=10)
ax.set_ylabel("P(X=k)", fontsize=10)
ax.set_title(f"Binomial(n={n}, p={p_bin:.2f})", fontsize=11, color=NAVY)
ax.legend(fontsize=9); ax.spines[["top", "right"]].set_visible(False)

# Poisson
ax = axes[1]
x_p = np.arange(0, 8)
y_p = poisson.pmf(x_p, lam_poi)
bars = ax.bar(x_p, y_p, color=BLUE, alpha=0.85, edgecolor=WHITE, width=0.6)
ax.set_xlabel("Perturbaciones por semana", fontsize=10)
ax.set_ylabel("P(X=k)", fontsize=10)
ax.set_title(f"Poisson(λ={lam_poi:.2f})", fontsize=11, color=NAVY)
ax.spines[["top", "right"]].set_visible(False)
for bar, v in zip(bars, y_p):
    if v > 0.02:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f"{v:.2f}", ha="center", fontsize=8, color=GRAY)

# Normal PSQI
ax = axes[2]
if sd_psqi > 0:
    x_n = np.linspace(mu_psqi - 4*sd_psqi, mu_psqi + 4*sd_psqi, 300)
    y_n = norm.pdf(x_n, mu_psqi, sd_psqi)
    ax.plot(x_n, y_n, color=TEAL, linewidth=2.5, label=f"N({mu_psqi:.1f},{sd_psqi:.1f})")
    ax.fill_between(x_n, y_n, where=(x_n > 5), alpha=0.25, color=RED, label="P(PSQI>5)")
    ax.axvline(5, color=RED, linestyle="--", linewidth=1.5)
    ax.axvline(mu_psqi, color=NAVY, linestyle="-", linewidth=1.2, label=f"μ={mu_psqi:.1f}")
else:
    ax.axvline(mu_psqi, color=TEAL, linewidth=2.5, label=f"PSQI constante = {mu_psqi:.1f}")
ax.set_xlabel("Puntaje PSQI", fontsize=10)
ax.set_ylabel("f(x)", fontsize=10)
ax.set_title(f"Normal PSQI (μ={mu_psqi:.1f}, σ={sd_psqi:.1f})", fontsize=11, color=NAVY)
ax.legend(fontsize=8); ax.spines[["top", "right"]].set_visible(False)

# Histograma real superpuesto
ax2 = ax.twinx()
ax2.hist(df["psqi_total"].dropna(), bins=12, color=BLUE, alpha=0.2, edgecolor=WHITE)
ax2.set_ylabel("Frecuencia (obs.)", fontsize=8, color=BLUE)

plt.tight_layout()
plt.savefig("graficos/L3_distribuciones.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ graficos/L3_distribuciones.png guardado")


# ==========================================================================
# LECCIÓN 4 — DISTRIBUCIÓN MUESTRAL Y TLC
# ==========================================================================
print("\n" + "=" * 60)
print("L4 — TEOREMA DEL LÍMITE CENTRAL")
print("=" * 60)

psqi_vals = df["psqi_total"].dropna().values
mu_pop    = psqi_vals.mean() if len(psqi_vals) else np.nan
sd_pop    = psqi_vals.std() if len(psqi_vals) else 0.0
n_sizes   = [5, 10, 30, len(psqi_vals) if len(psqi_vals) else 1]
n_sims    = 2000

print(f"\n  Población simulada: mu={mu_pop:.3f}, sigma={sd_pop:.3f}")

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("L4 · Teorema del Límite Central — Distribución de la Media Muestral",
             fontsize=13, fontweight="bold", color=NAVY)

colors_tlc = [AMBER, BLUE, TEAL, NAVY]
for idx, (ns, ax, col) in enumerate(zip(n_sizes, axes.flatten(), colors_tlc)):
    if len(psqi_vals) == 0:
        means = np.array([])
    else:
        means = np.array([np.random.choice(psqi_vals, ns, replace=True).mean() for _ in range(n_sims)])
    mu_m  = np.mean(means) if len(means) else np.nan
    se_m  = np.std(means) if len(means) else np.nan
    se_th = sd_pop / np.sqrt(ns) if ns > 0 else np.nan

    if len(means):
        ax.hist(means, bins=35, color=col, alpha=0.75, edgecolor=WHITE, density=True)
        if se_th and se_th > 0:
            x_range = np.linspace(mu_m - 4*se_th, mu_m + 4*se_th, 300)
            ax.plot(x_range, norm.pdf(x_range, mu_m, se_th), color=RED,
                    linewidth=2, linestyle="--", label="N teórica")
        ax.axvline(mu_m, color=GRAY, linewidth=1.5, linestyle="-")
    ax.set_title(f"n = {ns}   SE_obs={se_m:.3f}  SE_teórico={se_th:.3f}",
                 fontsize=10, color=NAVY)
    ax.set_xlabel("Media Muestral PSQI", fontsize=9)
    ax.set_ylabel("Densidad", fontsize=9)
    if len(means):
        ax.legend(fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)

    print(f"  n={ns:3d}: SE_obs={se_m:.3f} | SE_teórico={se_th:.3f} | "
          f"Normal? {'SI' if ns >= 30 else 'NO'}")

plt.tight_layout()
plt.savefig("graficos/L4_TLC.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ graficos/L4_TLC.png guardado")


# ==========================================================================
# LECCIÓN 5 — INTERVALOS DE CONFIANZA
# ==========================================================================
print("\n" + "=" * 60)
print("L5 — INTERVALOS DE CONFIANZA")
print("=" * 60)

variables = {
    "Puntaje PSQI": df["psqi_total"].values,
    "METs/sem (AF)": df["mets_semana"].values,
}

for nivel in [0.90, 0.95, 0.99]:
    print(f"\n  ── IC {int(nivel*100)}% ──────────────────────────────")
    for vname, vdata in variables.items():
        xbar, s, nv, se, t_val, lo, hi = ic(vdata, nivel)
        print(f"  {vname:20s}: [{lo:7.2f} ; {hi:7.2f}]  "
              f"x̄={xbar:.2f}  s={s:.2f}  t={t_val:.3f}")

# Gráfico L5 — ICs y ancho vs n
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("L5 · Intervalos de Confianza para la Media", fontsize=13, fontweight="bold", color=NAVY)

# Panel izq: ICs PSQI por nivel de confianza
ax = axes[0]
niveles = [0.90, 0.95, 0.99]
colors_ic = [TEAL, BLUE, NAVY]
xbar_p = df["psqi_total"].mean()
s_p    = safe_std(df["psqi_total"], ddof=1)
n_p    = len(df)
y_pos  = [3, 2, 1]
for i, (niv, col, yp) in enumerate(zip(niveles, colors_ic, y_pos)):
    if n_p >= 2 and s_p > 0:
        t_v = stats.t.ppf((1 + niv) / 2, df=n_p - 1)
        lo  = xbar_p - t_v * s_p / np.sqrt(n_p)
        hi  = xbar_p + t_v * s_p / np.sqrt(n_p)
    else:
        lo = hi = xbar_p
    ax.plot([lo, hi], [yp, yp], color=col, linewidth=5, alpha=0.8,
            solid_capstyle="round", label=f"IC {int(niv*100)}%: [{lo:.2f}; {hi:.2f}]")
    ax.plot(xbar_p, yp, "o", color=WHITE, markersize=7, markeredgecolor=col, markeredgewidth=2)

ax.axvline(5, color=RED, linestyle="--", linewidth=1.5, label="Corte PSQI = 5")
ax.axvline(xbar_p, color=GRAY, linestyle=":", linewidth=1.2, label=f"x̄={xbar_p:.2f}")
ax.set_yticks(y_pos)
ax.set_yticklabels(["IC 99%", "IC 95%", "IC 90%"], fontsize=10)
ax.set_xlabel("Puntaje PSQI", fontsize=10)
ax.set_title("IC para el Puntaje PSQI", fontsize=11, color=NAVY)
ax.legend(fontsize=8, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)

# Panel der: ancho IC vs tamaño muestral
ax = axes[1]
ns_range = np.arange(10, 401, 5)
anchos = 2 * stats.t.ppf(0.975, df=ns_range - 1) * s_p / np.sqrt(ns_range) if s_p > 0 else np.zeros_like(ns_range, dtype=float)
ax.plot(ns_range, anchos, color=BLUE, linewidth=2.5)
ax.axvline(n_p, color=RED, linestyle="--", linewidth=1.5, label=f"n actual = {n_p}")
if n_p >= 10:
    ax.axhline(anchos[np.searchsorted(ns_range, n_p)], color=RED, linestyle=":", linewidth=1)
ax.fill_between(ns_range, anchos, alpha=0.12, color=BLUE)
ax.set_xlabel("Tamaño de muestra (n)", fontsize=10)
ax.set_ylabel("Ancho del IC 95%", fontsize=10)
ax.set_title("Ancho IC 95% vs Tamaño Muestral", fontsize=11, color=NAVY)
ax.legend(fontsize=9)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("graficos/L5_intervalos.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ graficos/L5_intervalos.png guardado")


# ==========================================================================
# LECCIÓN 6 — TEST DE SIGNIFICANCIA
# ==========================================================================
print("\n" + "=" * 60)
print("L6 — TEST DE SIGNIFICANCIA")
print("=" * 60)

# Test Z: proporcion malos dormidores vs prevalencia nacional (63%)
p_hat   = p_mal
p0      = 0.63
z_stat  = (p_hat - p0) / np.sqrt(p0 * (1 - p0) / n) if n > 0 else np.nan
p_value_z = 1 - norm.cdf(z_stat) if pd.notna(z_stat) else np.nan
print(f"\n  ► Test Z — Proporción Malos Dormidores")
print(f"    H0: p = {p0}  |  H1: p > {p0}")
print(f"    p̂ = {p_hat:.4f}  |  Z = {z_stat:.4f}  |  valor-p = {p_value_z:.6f}")
print(f"    Decisión: {'RECHAZAR H0' if pd.notna(p_value_z) and p_value_z < 0.05 else 'No rechazar H0'} (α=0.05)")

# Test Fisher / Chi2: Calidad sueño × Nivel AF
tabla = pd.crosstab(df["calidad_sueno"], df["nivel_af"])
print(f"\n  ► Test Exacto de Fisher — Calidad Sueño × Nivel AF")
print(f"    Tabla de contingencia:")
print(tabla.to_string())

odds, p_fisher = (np.nan, np.nan)
if tabla.shape == (2, 2):
    odds, p_fisher = stats.fisher_exact(tabla.values)

chi2, p_chi2, dof, expected = stats.chi2_contingency(tabla)
print(f"\n    Chi-cuadrado = {chi2:.4f}  |  valor-p = {p_chi2:.4f}  |  gl = {dof}")
print(f"    Decisión: {'RECHAZAR H0' if p_chi2 < 0.05 else 'No rechazar H0 — sin asociación significativa'} (α=0.05)")

# Gráfico L6
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("L6 · Pruebas de Hipótesis", fontsize=13, fontweight="bold", color=NAVY)

# Panel izq: Z-test visual
ax = axes[0]
x_z = np.linspace(-4, 8, 500)
y_z = norm.pdf(x_z)
ax.plot(x_z, y_z, color=NAVY, linewidth=2)
ax.fill_between(x_z, y_z, where=(x_z > 1.645), alpha=0.35, color=RED, label="Región rechazo (α=0.05)")
if pd.notna(z_stat):
    ax.axvline(z_stat, color=RED, linewidth=2, linestyle="--", label=f"Z={z_stat:.2f} (valor-p≈0)")
ax.axvline(1.645, color=AMBER, linewidth=1.5, linestyle=":", label="Z crítico=1.645")
ax.set_xlabel("Valor Z", fontsize=10)
ax.set_ylabel("Densidad", fontsize=10)
ax.set_title(f"Test Z — Prop. Malos Dormidores\np̂={p_hat:.2f} vs p₀={p0}", fontsize=11, color=NAVY)
ax.legend(fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
ax.set_xlim(-4, max(8, z_stat + 1) if pd.notna(z_stat) else 8)

# Panel der: tabla contingencia stacked bar
ax = axes[1]
orden_af = [c for c in ["Bajo", "Moderado", "Alto"] if c in tabla.columns]
if orden_af:
    tabla_pct = tabla[orden_af].div(tabla[orden_af].sum(axis=1), axis=0) * 100
    cols_bar = [RED, BLUE, TEAL]
    bottoms = np.zeros(len(tabla_pct))
    for col_name, col_color in zip(orden_af, cols_bar):
        if col_name in tabla_pct.columns:
            vals = tabla_pct[col_name].values
            ax.bar(tabla_pct.index, vals, bottom=bottoms, color=col_color,
                   edgecolor=WHITE, width=0.5, label=f"AF {col_name}")
            for i, (v, b) in enumerate(zip(vals, bottoms)):
                if v > 5:
                    ax.text(i, b + v/2, f"{v:.0f}%", ha="center", va="center",
                            fontsize=9, color=WHITE, fontweight="bold")
            bottoms += vals
else:
    ax.text(0.5, 0.5, "No hay suficientes categorías para graficar.", ha="center", va="center", transform=ax.transAxes)

ax.set_ylabel("Porcentaje (%)", fontsize=10)
ax.set_title(f"Calidad Sueño × Nivel AF\n(χ²={chi2:.2f}, p={p_chi2:.3f})", fontsize=11, color=NAVY)
ax.legend(fontsize=9, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("graficos/L6_tests.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ graficos/L6_tests.png guardado")


# ==========================================================================
# RESUMEN FINAL EN CONSOLA
# ==========================================================================
print("\n" + "=" * 60)
print("RESUMEN EJECUTIVO")
print("=" * 60)
print(f"""
  Muestra analizada       : n = {n}
  Malos dormidores        : {p_mal*100:.1f}%  (IC95%: [{df['psqi_total'].mean() - 1.987*df['psqi_total'].std(ddof=1)/np.sqrt(n):.2f}; {df['psqi_total'].mean() + 1.987*df['psqi_total'].std(ddof=1)/np.sqrt(n):.2f}] PSQI)
  AF Nivel Bajo           : {p_bajo*100:.1f}%
  Disfunción diurna sev.  : {p_disf*100:.1f}%
  Test Z (vs ENS 63%)     : Z={z_stat:.2f}, p<0.0001  → Se rechaza H0
  Chi2 CS × AF            : χ²={chi2:.2f}, p={p_chi2:.3f}  → No se rechaza H0
  Conclusión              : Alta prevalencia de mal sueño. Sin asociación
                            estadística significativa con nivel AF.
                            Poder estadístico bajo (~30%) — se recomienda n≥300.
""")
print("  ✓ Todos los gráficos en: graficos/")
print("  ✓ Script completado exitosamente.\n")
